In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
production_company_schema = StructType(fields= [
                       StructField("companyId", IntegerType(), True),
                       StructField("companyName", StringType(), True)
])

In [0]:
production_company_df = spark.read \
                   .option("header", True) \
                   .schema(production_company_schema) \
                   .csv(f"{bronze_folder_path}/{v_file_date}/production_company")

In [0]:
display(production_company_df)

companyId,companyName
38833,Ian Bryce Productions
38944,Clavius Base
38956,Tandem Pictures
38957,4 1/2 Film
39043,Traveling Picture Show Company (TPSC)
39075,Strohberry Films
39077,Honora Productions
39121,Super Cool ManChu
39134,Kino Lorber
39226,Camellia Productions


In [0]:
production_company_df.count()

1046

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
production_company_renamed_df = add_ingestion_date(production_company_df) \
                            .withColumnsRenamed({"companyId": "company_Id", "companyName": "company_Name"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(production_company_renamed_df)

company_Id,company_Name,ingestion_date,environment,file_date
38833,Ian Bryce Productions,2026-09-11T04:38:07.415548Z,production,2024-12-30
38944,Clavius Base,2026-09-11T04:38:07.415548Z,production,2024-12-30
38956,Tandem Pictures,2026-09-11T04:38:07.415548Z,production,2024-12-30
38957,4 1/2 Film,2026-09-11T04:38:07.415548Z,production,2024-12-30
39043,Traveling Picture Show Company (TPSC),2026-09-11T04:38:07.415548Z,production,2024-12-30
39075,Strohberry Films,2026-09-11T04:38:07.415548Z,production,2024-12-30
39077,Honora Productions,2026-09-11T04:38:07.415548Z,production,2024-12-30
39121,Super Cool ManChu,2026-09-11T04:38:07.415548Z,production,2024-12-30
39134,Kino Lorber,2026-09-11T04:38:07.415548Z,production,2024-12-30
39226,Camellia Productions,2026-09-11T04:38:07.415548Z,production,2024-12-30


In [0]:
# overwrite_partition("movie_silver", "productions_companies", "file_date", v_file_date)

In [0]:
merge_delta_lake(production_company_renamed_df, "movie_silver", "productions_companies", "company_Id", "file_date")

In [0]:
# production_company_renamed_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.productions_companies")

In [0]:
display(spark.read.table("movie_silver.productions_companies"))

company_Id,company_Name,ingestion_date,environment,file_date
38833,Ian Bryce Productions,2026-09-11T04:38:08.618134Z,production,2024-12-30
38944,Clavius Base,2026-09-11T04:38:08.618134Z,production,2024-12-30
38956,Tandem Pictures,2026-09-11T04:38:08.618134Z,production,2024-12-30
38957,4 1/2 Film,2026-09-11T04:38:08.618134Z,production,2024-12-30
39043,Traveling Picture Show Company (TPSC),2026-09-11T04:38:08.618134Z,production,2024-12-30
39075,Strohberry Films,2026-09-11T04:38:08.618134Z,production,2024-12-30
39077,Honora Productions,2026-09-11T04:38:08.618134Z,production,2024-12-30
39121,Super Cool ManChu,2026-09-11T04:38:08.618134Z,production,2024-12-30
39134,Kino Lorber,2026-09-11T04:38:08.618134Z,production,2024-12-30
39226,Camellia Productions,2026-09-11T04:38:08.618134Z,production,2024-12-30


In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.productions_companies
GROUP BY file_date;

file_date,count(1)
2024-12-16,2997
2024-12-23,999
2024-12-30,1046


In [0]:
%sql
DESCRIBE EXTENDED movie_silver.productions_companies;

col_name,data_type,comment
company_Id,int,null
company_Name,string,null
ingestion_date,timestamp,null
environment,string,null
file_date,string,null
# Partition Information,,
# col_name,data_type,comment
file_date,string,null
,,
# Delta Statistics Columns,,


In [0]:
dbutils.notebook.exit("Success")